# 🕵️‍♂️ Agentic Sherlock AI — Google Colab GPU Inference Bridge

**A Hierarchical Cognitive Multi-Agent Framework for Cyber Attack Reconstruction & Digital Forensics**  
Team Number: **G1266** | Mentor: **Para Upendar**  

This notebook provides a cloud-hosted **T4/A100 GPU Inference Microservice** for your local Sherlock AI desktop application (`sherf`).  
It loads **Qwen2.5-7B-Instruct** in 4-bit NF4 precision with dynamic PEFT LoRA adapter switching for the 5 domain agents (`network`, `memory`, `disk`, `logs`, `registry`), and serves the Supervisor & Case Chatbot on the base instruction model with structured JSON contracts.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

# 2. Install Required Dependencies
!pip install -q torch transformers peft bitsandbytes accelerate fastapi uvicorn pydantic nest_asyncio

In [ ]:
import os, sys, time, json, re, getpass, glob, zipfile
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Foundation Base Model for SherlockAI Domain LoRAs (hidden_size: 4096)
BASE_MODEL_ID = "Qwen/Qwen3-8B"
print(f"[1/4] Loading Tokenizer for {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print(f"[2/4] Loading 4-bit Quantized Base Model ({BASE_MODEL_ID}) onto GPU...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.eval()

print(f"[3/4] Unpacking and Locating PEFT LoRA Adapters...")
# Check for sherlock_adapters.zip or models.zip
for zip_cand in ["/content/sherlock_adapters.zip", "/content/models.zip"]:
    if os.path.exists(zip_cand):
        print(f"  Extracting {zip_cand}...")
        with zipfile.ZipFile(zip_cand, 'r') as zf:
            zf.extractall("/content")

candidate_roots = [
    "/content/sherlock_adapters",
    "/content/models",
    "/content"
]
adapters_root = "/content/sherlock_adapters"
for c in candidate_roots:
    if os.path.exists(f"{c}/disk/adapter_config.json") or os.path.exists(f"{c}/memory/adapter_config.json"):
        adapters_root = c
        break

loaded_adapters = []
peft_model = None
qwen3_domains = ["base_cybersecurity", "memory", "disk", "logs", "registry"]

for domain in qwen3_domains:
    p = f"{adapters_root}/{domain}"
    if os.path.exists(f"{p}/adapter_config.json"):
        try:
            if peft_model is None:
                peft_model = PeftModel.from_pretrained(base_model, p, adapter_name=domain)
            else:
                peft_model.load_adapter(p, adapter_name=domain)
            loaded_adapters.append(domain)
            print(f"  ✅ Mounted Qwen3 adapter: {domain}")
        except Exception as e:
            print(f"  ⚠️ Could not mount adapter '{domain}': {e}")

inference_model = peft_model if peft_model is not None else base_model

# Dedicated Network Model (Qwen2.5-7B) for Network Adapter
network_model = None
network_tokenizer = None
NETWORK_BASE_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
net_p = f"{adapters_root}/network"
if os.path.exists(f"{net_p}/adapter_config.json"):
    print(f"\n[4/4] Mounting Network Adapter ({NETWORK_BASE_ID})...")
    try:
        net_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        net_base = AutoModelForCausalLM.from_pretrained(
            NETWORK_BASE_ID,
            quantization_config=net_bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        net_base.eval()
        network_model = PeftModel.from_pretrained(net_base, net_p, adapter_name="network")
        network_tokenizer = AutoTokenizer.from_pretrained(NETWORK_BASE_ID, trust_remote_code=True)
        loaded_adapters.append("network")
        print(f"  ✅ Mounted dedicated Qwen2.5 adapter: network")
    except Exception as err:
        print(f"  ℹ️ Dedicated network model skipped or VRAM constrained: {err}")
        print(f"  Network domain analysis will execute with expert forensic prompt on {BASE_MODEL_ID}.")
        loaded_adapters.append("network (prompt-grounded)")

print(f"\n🔥 Model Setup Complete! Active Loaded Adapters: {loaded_adapters}")


In [ ]:
# Authentication Configuration
auth_token = os.environ.get("SHERLOCK_REMOTE_TOKEN", "")
if not auth_token:
    user_input_token = getpass.getpass("Enter SHERLOCK_REMOTE_TOKEN to secure endpoint (or leave blank to disable auth): ").strip()
    if user_input_token:
        auth_token = user_input_token
        os.environ["SHERLOCK_REMOTE_TOKEN"] = auth_token
        print("API Authentication ENABLED.")
    else:
        print("API Authentication DISABLED (Development Mode).")
else:
    print("Using pre-configured SHERLOCK_REMOTE_TOKEN.")

In [ ]:
import asyncio, time, json, re
from fastapi import FastAPI, Header, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Any, Dict, List

inference_lock = asyncio.Lock()

app = FastAPI(
    title="SherlockAI Remote GPU Inference Bridge",
    version="2.0.0",
    description="Provides GPU-accelerated Qwen multi-LoRA inference for digital forensic agents"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class InferenceRequest(BaseModel):
    domain: str
    evidence_type: str
    evidence: Any
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.1

def verify_auth(x_sherlock_token: Optional[str] = Header(None)):
    if auth_token and x_sherlock_token != auth_token:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid or missing X-Sherlock-Token")

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "hardware": "Google Colab Cloud GPU (Nvidia)",
        "base_model": BASE_MODEL_ID,
        "loaded_adapters": loaded_adapters,
        "peft_enabled": peft_model is not None or network_model is not None
    }

@app.get("/api/v1/adapters")
def list_adapters():
    active = None
    if peft_model is not None:
        active = getattr(peft_model, "active_adapter", None)
    return {
        "loaded_adapters": loaded_adapters,
        "active_adapter": active,
        "available_domains": ["network", "memory", "disk", "logs", "registry", "base_cybersecurity", "supervisor"]
    }

@app.post("/api/v1/inference")
async def run_inference(req: InferenceRequest, x_sherlock_token: Optional[str] = Header(None)):
    verify_auth(x_sherlock_token)
    domain = req.domain.lower()
    target_adapter = "base_cybersecurity" if domain in ["supervisor", "base_cybersecurity"] else domain

    async with inference_lock:
        active_model = inference_model
        active_tok = tokenizer
        adapter_used = "base_model"

        if domain == "network" and network_model is not None:
            active_model = network_model
            active_tok = network_tokenizer
            adapter_used = "network (Qwen2.5-7B LoRA)"
        elif peft_model is not None and target_adapter in loaded_adapters:
            peft_model.set_adapter(target_adapter)
            adapter_used = target_adapter
        elif domain == "network":
            adapter_used = f"{BASE_MODEL_ID} (Domain Specialist)"
        else:
            adapter_used = f"{BASE_MODEL_ID} (Base)"
        
        system_prompt = (
            f"You are the SherlockAI {domain.capitalize()} Forensic Analysis Agent. "
            "Analyze the provided forensic telemetry with zero hallucination. "
            "Return ONLY a valid JSON object matching this schema:\n"
            "{\n"
            '  "classification": "BENIGN" | "SUSPICIOUS" | "MALICIOUS" | "INCONCLUSIVE",\n'
            '  "risk_level": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",\n'
            '  "confidence": 0.0 to 1.0,\n'
            '  "reasoning": "string explaining causal forensic deduction",\n'
            '  "key_evidence": ["specific indicator string", ...],\n'
            '  "mitre_technique": "Txxxx.xxx or N/A"\n'
            "}"
        )
        
        evidence_str = json.dumps(req.evidence, indent=2, default=str) if not isinstance(req.evidence, str) else req.evidence
        user_prompt = f"Domain: {domain}\nEvidence Type: {req.evidence_type}\nTelemetry:\n{evidence_str[:4000]}"
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        prompt_text = active_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = active_tok(prompt_text, return_tensors="pt").to(active_model.device)
        
        with torch.no_grad():
            outputs = active_model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                do_sample=False,
                pad_token_id=active_tok.eos_token_id
            )
        
        generated_text = active_tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Extract JSON from output
        json_match = re.search(r"\{.*\}", generated_text, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group(0))
                parsed["model_info"] = {
                    "provider": "Google Colab GPU",
                    "base_model": BASE_MODEL_ID,
                    "adapter_used": adapter_used
                }
                return parsed
            except Exception:
                pass
        
        return {
            "classification": "INCONCLUSIVE",
            "risk_level": "LOW",
            "confidence": 0.5,
            "reasoning": generated_text.strip(),
            "key_evidence": [],
            "raw_output": generated_text,
            "model_info": {"provider": "Google Colab GPU", "base_model": BASE_MODEL_ID, "adapter_used": adapter_used}
        }

print("✅ FastAPI Microservice Endpoints Registered.")


In [ ]:
# 5. Launch Cloudflare Tunnel and Start Service
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, time, threading, re

def start_server():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(3)

print("Starting Cloudflare tunnel bridge...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
for line in tunnel_proc.stderr:
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("=" * 68)
        print(f"🚀 COLAB GPU INFERENCE TUNNEL ACTIVE: {tunnel_url}")
        print("=" * 68)
        print("Paste this URL into your Sherlock AI settings or sherf/backend/.env:")
        print(f'COLAB_GPU_INFERENCE_URL="{tunnel_url}"')
        if os.environ.get("SHERLOCK_REMOTE_TOKEN"):
            print(f'SHERLOCK_REMOTE_TOKEN="{os.environ.get("SHERLOCK_REMOTE_TOKEN")}"')
        print("=" * 68)
        print("Service is LIVE and serving neural inferences on GPU. Keep this cell running!")
        break

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping tunnel...")
    tunnel_proc.terminate()
